# Role compression and cold-start calibration

Leakage-safe analysis of existing outer-fold minutes predictions. No model fitting or optimizer changes.

In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB = ROOT / "data/db/euroleague.duckdb"
MINUTES_PATH = ROOT / "data/derived/phase4b/components/p4b__minutes__catboost__role_older_history/outer_predictions.parquet"
FEATURE_PATH = ROOT / "data/derived/phase6c/research/feature_frame.parquet"

con = duckdb.connect()
minutes = con.execute("SELECT * FROM read_parquet(?)", [str(MINUTES_PATH)]).df()
features = con.execute("SELECT * FROM read_parquet(?)", [str(FEATURE_PATH)]).df()
con.close()

feature_cols = [
    "model_row_id", "season_start_year", "season_games_before",
    "last_5_minutes_avg", "minutes_ewma", "season_minutes_avg_before",
    "target_game_time", "team_id", "player_id", "target_minutes",
]
audit = minutes.merge(
    features[feature_cols], on="model_row_id", validate="one_to_one",
    suffixes=("", "_feature"),
)
audit["matchday"] = audit["round_number"].astype(int)

history = features[[
    "season_start_year", "player_id", "team_id", "target_game_time", "target_minutes"
]].copy()
history = history[pd.to_numeric(history.target_minutes, errors="coerce").gt(0)]
history = history.sort_values(["player_id", "season_start_year", "target_game_time"])
prior = history.groupby(["player_id", "season_start_year"], as_index=False).agg(
    previous_season_games=("target_minutes", "size"),
    previous_season_mpg=("target_minutes", "mean"),
    previous_team_id=("team_id", "last"),
)
prior_last5 = (
    history.groupby(["player_id", "season_start_year"]).tail(5)
    .groupby(["player_id", "season_start_year"], as_index=False)
    .agg(previous_season_last5_mpg=("target_minutes", "mean"))
)
prior = prior.merge(
    prior_last5, on=["player_id", "season_start_year"], validate="one_to_one"
).rename(columns={"season_start_year": "previous_season_start_year"})
audit["previous_season_start_year"] = audit["season_start_year"] - 1
audit = audit.merge(
    prior, on=["player_id", "previous_season_start_year"],
    how="left", validate="many_to_one",
)
audit["returning_player"] = audit.previous_season_mpg.notna()

def role_label(minutes_value):
    if pd.isna(minutes_value):
        return "UNKNOWN"
    if minutes_value >= 28:
        return "STAR / PRIMARY"
    if minutes_value >= 22:
        return "STARTER"
    if minutes_value >= 14:
        return "ROTATION"
    return "BENCH / LIMITED"

audit["historical_role"] = audit.previous_season_mpg.map(role_label)
assert set(audit.season) == {"E2023", "E2024", "E2025"}
assert audit.model_row_id.is_unique
assert (audit.actual_minutes > 0).all()
assert audit[["predicted_minutes", "actual_minutes"]].notna().all().all()
print({
    "rows": len(audit),
    "returning_md1": len(audit[audit.returning_player & audit.matchday.eq(1)]),
    "seasons": sorted(audit.season.unique()),
})

{'rows': 22399, 'returning_md1': 419, 'seasons': ['E2023', 'E2024', 'E2025']}


In [2]:
def anchor_values(frame, kind):
    if kind == "previous_mpg":
        return frame.previous_season_mpg
    if kind == "previous_75_last5_25":
        return 0.75 * frame.previous_season_mpg + 0.25 * frame.previous_season_last5_mpg
    if kind == "previous_50_last5_50":
        return 0.50 * frame.previous_season_mpg + 0.50 * frame.previous_season_last5_mpg
    if kind == "existing_last5":
        return frame.last_5_minutes_avg
    if kind == "previous_75_existing_last5_25":
        return 0.75 * frame.previous_season_mpg + 0.25 * frame.last_5_minutes_avg
    raise KeyError(kind)

def apply_schedule(frame, w0, w1, anchor_kind):
    corrected = frame.predicted_minutes.astype(float).copy()
    anchor = anchor_values(frame, anchor_kind)
    weights = np.select(
        [frame.season_games_before.eq(0), frame.season_games_before.eq(1)],
        [w0, w1],
        default=0.0,
    )
    eligible = frame.returning_player & anchor.notna()
    corrected.loc[eligible] = (
        (1 - weights[eligible]) * corrected[eligible]
        + weights[eligible] * anchor[eligible]
    )
    return corrected.clip(0, 40)

def mae(actual, predicted):
    return float(np.mean(np.abs(np.asarray(predicted) - np.asarray(actual))))

train = audit[
    audit.season.isin(["E2023", "E2024"])
    & audit.returning_player
    & audit.matchday.le(3)
].copy()
validation = audit[
    audit.season.eq("E2025")
    & audit.returning_player
].copy()

grid_rows = []
anchor_kinds = [
    "previous_mpg",
    "previous_75_last5_25",
    "previous_50_last5_50",
    "existing_last5",
    "previous_75_existing_last5_25",
]
for anchor_kind in anchor_kinds:
    for w0 in np.arange(0, 0.81, 0.05):
        for w1 in np.arange(0, min(0.41, w0 + 0.001), 0.05):
            predicted = apply_schedule(train, float(w0), float(w1), anchor_kind)
            grid_rows.append({
                "anchor": anchor_kind,
                "w0": round(float(w0), 2),
                "w1": round(float(w1), 2),
                "train_md1_3_mae": mae(train.actual_minutes, predicted),
                "train_md1_mae": mae(
                    train.loc[train.matchday.eq(1), "actual_minutes"],
                    predicted.loc[train.matchday.eq(1)],
                ),
            })

grid = pd.DataFrame(grid_rows)
baseline_train = mae(train.actual_minutes, train.predicted_minutes)
grid["improvement"] = baseline_train - grid.train_md1_3_mae
best = grid.sort_values(
    ["train_md1_3_mae", "train_md1_mae", "w0", "w1"],
    ascending=[True, True, True, True],
).iloc[0]
print("training baseline MD1-3 MAE", round(baseline_train, 4))
print(grid.sort_values("train_md1_3_mae").head(12).round(4).to_string(index=False))
print()
print("chosen", best.to_dict())

CHOSEN_ANCHOR = str(best.anchor)
CHOSEN_W0 = float(best.w0)
CHOSEN_W1 = float(best.w1)
audit["corrected_minutes"] = apply_schedule(
    audit, CHOSEN_W0, CHOSEN_W1, CHOSEN_ANCHOR
)

training baseline MD1-3 MAE 4.9877
                       anchor   w0   w1  train_md1_3_mae  train_md1_mae  improvement
                 previous_mpg 0.30 0.00           4.9623         5.6324       0.0254
                 previous_mpg 0.35 0.00           4.9629         5.6266       0.0248
                 previous_mpg 0.25 0.00           4.9630         5.6421       0.0247
                 previous_mpg 0.40 0.00           4.9638         5.6220       0.0239
                 previous_mpg 0.30 0.05           4.9651         5.6324       0.0226
                 previous_mpg 0.45 0.00           4.9656         5.6176       0.0222
                 previous_mpg 0.35 0.05           4.9657         5.6266       0.0221
                 previous_mpg 0.25 0.05           4.9658         5.6421       0.0219
                 previous_mpg 0.20 0.00           4.9662         5.6593       0.0215
previous_75_existing_last5_25 0.35 0.00           4.9663         5.6331       0.0214
previous_75_existing_last5_25 

In [3]:
def metric_record(frame, predicted_column):
    error = frame[predicted_column] - frame.actual_minutes
    return {
        "N": len(frame),
        "MAE": float(error.abs().mean()),
        "Bias": float(error.mean()),
        "Predicted SD": float(frame[predicted_column].std(ddof=0)),
        "Actual SD": float(frame.actual_minutes.std(ddof=0)),
    }

window_masks = {
    "MD1": audit.matchday.eq(1),
    "MD1-3": audit.matchday.le(3),
    "MD2-3": audit.matchday.between(2, 3),
    "MD4-5": audit.matchday.between(4, 5),
    "Later (MD6+)": audit.matchday.ge(6),
}
metric_rows = []
for population, population_mask in [
    ("E2025 validation", audit.season.eq("E2025")),
    ("All frozen outer", pd.Series(True, index=audit.index)),
]:
    for window, window_mask in window_masks.items():
        sample = audit[population_mask & window_mask]
        before = metric_record(sample, "predicted_minutes")
        after = metric_record(sample, "corrected_minutes")
        metric_rows.append({
            "Population": population,
            "Window": window,
            "N": before["N"],
            "MAE before": before["MAE"],
            "MAE after": after["MAE"],
            "MAE delta": after["MAE"] - before["MAE"],
            "Bias before": before["Bias"],
            "Bias after": after["Bias"],
            "Pred SD before": before["Predicted SD"],
            "Pred SD after": after["Predicted SD"],
            "Actual SD": before["Actual SD"],
        })
metrics = pd.DataFrame(metric_rows)
print(metrics.round(3).to_string(index=False))

role_order = ["STAR / PRIMARY", "STARTER", "ROTATION", "BENCH / LIMITED"]
role_rows = []
for population, population_mask in [
    ("E2025 validation", audit.season.eq("E2025")),
    ("All frozen outer", pd.Series(True, index=audit.index)),
]:
    for window, window_mask in window_masks.items():
        sample = audit[
            population_mask & window_mask & audit.returning_player
        ]
        for role in role_order:
            group = sample[sample.historical_role.eq(role)]
            if group.empty:
                continue
            role_rows.append({
                "Population": population,
                "Window": window,
                "Role": role,
                "N": len(group),
                "Prior MPG": group.previous_season_mpg.mean(),
                "Pred before": group.predicted_minutes.mean(),
                "Pred after": group.corrected_minutes.mean(),
                "Actual": group.actual_minutes.mean(),
                "Bias before": (group.predicted_minutes - group.actual_minutes).mean(),
                "Bias after": (group.corrected_minutes - group.actual_minutes).mean(),
            })
role_metrics = pd.DataFrame(role_rows)
print()
print(role_metrics[
    role_metrics.Window.isin(["MD1", "MD2-3", "MD4-5", "Later (MD6+)"])
].round(2).to_string(index=False))

spread_rows = []
for population in role_metrics.Population.unique():
    for window in window_masks:
        subset = role_metrics[
            role_metrics.Population.eq(population)
            & role_metrics.Window.eq(window)
        ].set_index("Role")
        if {"STAR / PRIMARY", "BENCH / LIMITED"}.issubset(subset.index):
            spread_rows.append({
                "Population": population,
                "Window": window,
                "Before star-bench spread": (
                    subset.loc["STAR / PRIMARY", "Pred before"]
                    - subset.loc["BENCH / LIMITED", "Pred before"]
                ),
                "After star-bench spread": (
                    subset.loc["STAR / PRIMARY", "Pred after"]
                    - subset.loc["BENCH / LIMITED", "Pred after"]
                ),
                "Actual star-bench spread": (
                    subset.loc["STAR / PRIMARY", "Actual"]
                    - subset.loc["BENCH / LIMITED", "Actual"]
                ),
            })
spreads = pd.DataFrame(spread_rows)
print()
print(spreads.round(2).to_string(index=False))

      Population       Window     N  MAE before  MAE after  MAE delta  Bias before  Bias after  Pred SD before  Pred SD after  Actual SD
E2025 validation          MD1   151       5.382      5.216     -0.166        0.292       0.404           4.079          4.402      7.588
E2025 validation        MD1-3   563       4.629      4.577     -0.053       -0.045       0.019           5.278          5.294      7.553
E2025 validation        MD2-3   412       4.354      4.342     -0.011       -0.168      -0.122           5.644          5.573      7.540
E2025 validation        MD4-5   426       4.545      4.560      0.015       -0.721      -0.714           5.731          5.729      7.676
E2025 validation Later (MD6+)  7527       4.449      4.449      0.000       -0.759      -0.753           6.056          6.051      7.962
All frozen outer          MD1   445       5.645      5.513     -0.132       -0.637      -0.395           4.268          4.602      8.184
All frozen outer        MD1-3  1599      

In [4]:
from src.live.minutes import (
    attach_previous_season_minutes,
    calibrate_cold_start_minutes,
    expected_role_label,
)

live = duckdb.connect(str(DB), read_only=True)
latest_run = live.execute("""
    SELECT prediction_run_id, prediction_generated_at, season_matchday
    FROM live_prediction_runs
    WHERE season_code='E2026' AND status IN ('SUCCEEDED', 'PARTIAL')
    ORDER BY prediction_generated_at DESC
    LIMIT 1
""").fetchone()

if latest_run is None:
    e2026_sample = pd.DataFrame()
    live.close()
    print("No successful/partial E2026 live prediction is stored; UI rendering is checked separately.")
else:
    run_id, generated_at, live_matchday = latest_run
    scenario_row = live.execute("""
        SELECT scenario_id
        FROM live_prediction_scenarios
        WHERE prediction_run_id=? AND scenario_status='GENERATED'
        ORDER BY scenario_id
        LIMIT 1
    """, [run_id]).fetchone()
    scenario_id = scenario_row[0] if scenario_row else None
    current = live.execute("""
        SELECT player_name, canonical_player_id AS player_id,
               canonical_team_id AS team_id, season_code AS season,
               adjusted_expected_minutes AS baseline_expected_minutes,
               season_games_before
        FROM live_player_predictions
        WHERE prediction_run_id=? AND scenario_id=?
          AND prediction_status='SCORED'
    """, [run_id, scenario_id]).df() if scenario_id else pd.DataFrame()
    live.close()
    if current.empty:
        e2026_sample = pd.DataFrame()
        print("The latest E2026 run has no scored generated scenario; UI rendering is checked separately.")
    else:
        current = attach_previous_season_minutes(
            current, database_path=DB, cutoff=generated_at
        )
        current = calibrate_cold_start_minutes(current)
        current["old_predicted_minutes"] = current["uncalibrated_expected_minutes"]
        current["corrected_predicted_minutes"] = current["baseline_expected_minutes"]
        current["expected_role"] = current.corrected_predicted_minutes.map(expected_role_label)
        target_midpoints = {
            "STAR / PRIMARY": 30.0, "STARTER": 25.0,
            "ROTATION": 18.0, "BENCH / LIMITED": 11.0,
        }
        current["sample_distance"] = [
            abs(minutes - target_midpoints[role])
            for minutes, role in zip(
                current.corrected_predicted_minutes, current.expected_role, strict=True
            )
        ]
        e2026_sample = (
            current.sort_values("sample_distance").groupby("expected_role", as_index=False).head(1)
            [[
                "player_name", "season_games_before", "previous_season_minutes_avg",
                "old_predicted_minutes", "corrected_predicted_minutes", "expected_role",
            ]]
            .sort_values("corrected_predicted_minutes", ascending=False)
        )
        assert e2026_sample.expected_role.nunique() == 4
        print({
            "prediction_run_id": run_id, "scenario_id": scenario_id,
            "matchday": live_matchday, "rows": len(current),
        })
        print(e2026_sample.round(2).to_string(index=False))

{'prediction_run_id': '211cfcbd-8b09-58c0-b79d-b893a328fb4c', 'scenario_id': 'RESOLVED', 'matchday': 1, 'rows': 243}
       player_name  season_games_before  previous_season_minutes_avg  old_predicted_minutes  corrected_predicted_minutes   expected_role
Derrick Alston Jr.                    0                        22.43                  34.73                        31.04  STAR / PRIMARY
   Justin Robinson                    0                        18.97                  27.54                        24.97         STARTER
     Lamar Stevens                    0                        19.96                  17.10                        17.96        ROTATION
    Moustapha Fall                    0                         6.72                  12.89                        11.04 BENCH / LIMITED


## Current preparation context smoke check

Official boxes are retained in separate preparation tables with source artifacts and existing player IDs. Live inference blends at most 20% recent minutes evidence into the EuroLeague baseline; SuperCup/domestic games receive more weight than friendlies. The two cells below refresh current predictions with frozen artifacts, then compare two affected teams with the overlay disabled. They do not rerun the historical analysis or train models.

In [ ]:
from pathlib import Path
from datetime import datetime, UTC
from dataclasses import asdict
import json
import sys
import duckdb
import pandas as pd

PREP_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PREP_ROOT) not in sys.path:
    sys.path.insert(0, str(PREP_ROOT))
PREP_DB = PREP_ROOT / 'data/db/euroleague.duckdb'
from src.live.prediction import (
    run_live_prediction, predict_fantasy_points, load_decision_file,
    build_prediction_scenarios, load_and_validate_frozen_artifacts,
    load_and_validate_probabilistic_artifact, load_and_validate_predictive_uplift_artifact,
)

def prep_counts():
    with duckdb.connect(str(PREP_DB), read_only=True) as c:
        return {t: c.execute(f'SELECT count(*) FROM {t}').fetchone()[0]
                for t in ('players', 'games', 'player_game_stats')}

prep_before = prep_counts()
prep_cutoff = datetime.now(UTC)
prep_bundle, prep_validation = load_and_validate_frozen_artifacts()
prep_prob, prep_prob_validation = load_and_validate_probabilistic_artifact()
prep_uplift, prep_uplift_validation = load_and_validate_predictive_uplift_artifact()
print('Refreshing current predictions with frozen models...', flush=True)
prep_result = run_live_prediction(
    'E2026', database_path=PREP_DB, refresh=False, cutoff=prep_cutoff,
    model_bundle=prep_bundle, artifact_validation=prep_validation,
    probabilistic_bundle=prep_prob, probabilistic_artifact_validation=prep_prob_validation,
    predictive_bundle=prep_uplift, predictive_artifact_validation=prep_uplift_validation,
)
assert prep_result.status in ('SUCCEEDED', 'PARTIAL'), asdict(prep_result)
print(prep_result.status, prep_result.prediction_run_id)


In [2]:
with duckdb.connect(str(PREP_DB), read_only=True) as c:
    prep_slate = c.execute("SELECT * FROM current_upcoming_player_slate_v1 WHERE season='E2026'").df()
    prep_sources = c.execute('''SELECT g.source, count(DISTINCT g.preparation_game_id) AS games,
        count(*) AS player_rows, max(g.game_date) AS newest_game
        FROM preparation_games g JOIN preparation_player_stats s USING(preparation_game_id)
        GROUP BY g.source ORDER BY g.source''').df()
    prep_invalid = c.execute('''SELECT count(*) FROM preparation_player_stats
        WHERE points <> 2*two_points_made + 3*three_points_made + free_throws_made
        OR two_points_made > two_points_attempted OR three_points_made > three_points_attempted
        OR free_throws_made > free_throws_attempted''').fetchone()[0]
assert prep_invalid == 0
assert prep_counts() == prep_before
assert prep_slate.preparation_games.gt(0).any()
assert pd.to_datetime(prep_slate.preparation_latest_game, utc=True).max().date().isoformat() == '2026-09-20'
assert prep_slate.preparation_weight.max() <= .20
prep_output = pd.read_csv(prep_result.csv_path)
prep_teams = prep_slate.loc[prep_slate.preparation_games.gt(0)].groupby('team_id').preparation_games.sum().nlargest(2).index
prep_subset = prep_slate[prep_slate.team_id.isin(prep_teams)].copy()
prep_scenario = build_prediction_scenarios(prep_subset, load_decision_file(None))[0]
assert not prep_scenario.unresolved_players
prep_subset['preparation_weight'] = 0.0
with duckdb.connect(str(PREP_DB), read_only=True):
    prep_baseline, _, _ = predict_fantasy_points(
        prep_subset, prep_scenario, prep_bundle, prep_validation, prep_prob, prep_prob_validation,
        prep_uplift, prep_uplift_validation, database_path=PREP_DB,
        cutoff=prep_cutoff, fantasy_matchday=1,
    )
prep_comparison = prep_output.merge(
    prep_baseline[['player_id', 'adjusted_expected_minutes', 'expected_role', 'phase6c_expected_fp']],
    on='player_id', suffixes=('', '_without_context'), validate='many_to_one',
)
prep_comparison = prep_comparison[prep_comparison.preparation_games.gt(0) & prep_comparison.prediction_status.eq('SCORED')].copy()
prep_comparison['prediction_change'] = prep_comparison.expected_fp - prep_comparison.phase6c_expected_fp_without_context
assert prep_comparison.prediction_change.abs().gt(.001).sum() >= 3
assert (prep_comparison.adjusted_expected_minutes - prep_comparison.adjusted_expected_minutes_without_context).abs().gt(.001).sum() >= 3
prep_sample_columns = ['player_name', 'preparation_games', 'preparation_latest_game',
    'adjusted_expected_minutes_without_context', 'adjusted_expected_minutes',
    'preparation_starter_rate', 'preparation_expected_starter_rate', 'expected_role',
    'preparation_role_change', 'expected_fp', 'prediction_change']
prep_sample = prep_comparison.sort_values('prediction_change', key=lambda s: s.abs(), ascending=False)[prep_sample_columns].head(6)
prep_report = {'sources': json.loads(prep_sources.to_json(orient='records', date_format='iso')),
    'games_added': int(prep_sources.games.sum()), 'player_rows_added': int(prep_sources.player_rows.sum()),
    'prediction_run_id': prep_result.prediction_run_id, 'prediction_status': prep_result.status,
    'players_using_context': int(prep_output.preparation_games.fillna(0).gt(0).sum()),
    'historical_tables_and_player_count_unchanged': prep_counts() == prep_before,
    'sample': json.loads(prep_sample.to_json(orient='records')), 'smoke_passed': True}
(PREP_ROOT / 'reports/preparation_smoke.json').write_text(json.dumps(prep_report, indent=2))
print(prep_sources.to_string(index=False))
print(prep_sample.round(3).to_string(index=False))
print({k: v for k, v in prep_report.items() if k not in ('sources', 'sample')})

    source  games  player_rows               newest_game
       acb      3           72 2026-09-20 20:00:00+03:00
       bbl      1           24 2026-09-20 19:00:00+03:00
       lba      2           48 2026-09-20 18:00:00+03:00
       lkl      2           48 2026-09-20 16:50:00+03:00
       lnb      1           22 2026-09-20 18:00:00+03:00
   lnb_pre      5          117 2026-09-13 19:00:00+03:00
olympiacos      2           48 2026-09-19 21:00:00+03:00
       vtb      3           71 2026-09-20 19:00:00+03:00
        player_name  preparation_games   preparation_latest_game  adjusted_expected_minutes_without_context  adjusted_expected_minutes  preparation_starter_rate  preparation_expected_starter_rate expected_role preparation_role_change  expected_fp  prediction_change
    Azuolas Tubelis                  3 2026-09-20T16:50:00+03:00                                     19.517                     18.802                     1.000                              1.000      ROTATION           T